<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/03_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
# Install Unsloth and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken

In [ ]:
import os, sys
import torch
from google.colab import drive
from google.colab import userdata
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
from vector_store import RailVectorVault
from embed import RailEmbedder
from generator import RailDataGenerator

# Mount Drive to access your Vector DB and save the model
drive.mount('/content/drive')
api_key = userdata.get('GEMINI_API_KEY')
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"
TRAIN_DATA_PATH = "/content/rail-safety-ai/data/training/rail_dataset.jsonl"

In [ ]:
#
# os.path.dirname("/content/rail-safety-ai/data/training/rail_dataset.jsonl")
os.makedirs(os.path.dirname("/content/rail-safety-ai/data/training/rail_dataset.jsonl"), exist_ok=True)

In [ ]:
import importlib
import generator
importlib.reload(generator)
from generator import RailDataGenerator

In [ ]:
# Initialize the Librarian (Vault) and the Teacher (Generator)
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)


model_id = "gemini-2.5-flash"

url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent"

generator = RailDataGenerator(api_url = url, #"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-09-2025:generateContent",
                              api_key = api_key,
                              vault_instance=vault)



In [ ]:
# Generate the synthetic dataset
# 1 sample took about 30 seconds
res_dir = generator.create_dataset(num_samples=100, output_path=TRAIN_DATA_PATH)

In [ ]:
!wc -l /content/rail-safety-ai/data/training/rail_dataset.jsonl

In [ ]:
from unsloth import FastLanguageModel

model_id = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_id = model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: The "Staff" choice for balancing capacity vs. memory
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for Unsloth
    bias = "none",    # Optimized to "none" for Unsloth
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
import json
from datasets import Dataset

def formatting_prompts_func(examples):
    instructions = examples["question"]
    thoughts = examples["thinking"]
    answers = examples["answer"]
    texts = []

    for instruction, thought, answer in zip(instructions, thoughts, answers):
        # We wrap the thinking and answer together as the 'Assistant' response
        full_response = f"[THINKING PROCESS]\n{thought}\n\n[ANSWER]\n{answer}"

        messages = [
            {"role": "system", "content": "You are a Senior FRA Safety Consultant. Use a 4-Phase Thinking Process."},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": full_response},
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts, }

# Load the JSONL we just generated
import pandas as pd
df = pd.read_json(TRAIN_DATA_PATH, lines=True)
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched = True,)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Small step count for demonstration
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()


In [ ]:
model.save_pretrained_lora("/content/drive/MyDrive/rail_ai_project/rail_safety_adapters")
tokenizer.save_pretrained("/content/drive/MyDrive/rail_ai_project/rail_safety_adapters")